# Archivo en el cual Santino Lui toquetea

### Configuración del entorno y verificación de integridad de los datos crudos

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Agregar la raíz del proyecto al path para importar desde src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.auxiliar_functions import (
    RAW_DATA_DIR,
    DATA_DIR,
    SEMILLA,
    huella,
    verificar_checksums,
)

# 1. Verificar integridad SHA-256 de los 3 archivos Excel crudos
verificar_checksums()

Verificación de integridad completada: 3 archivo(s) coinciden.


True

### Comparar las hojas existentes entre los 3 archivos Excel

In [2]:
# Rutas de los 3 archivos Excel en data/raw/
ruta_2020_2022 = RAW_DATA_DIR / "Totalizadores Planta 2020_2022.xlsx"
ruta_2021_2023 = RAW_DATA_DIR / "Totalizadores Planta - 2021_2023.xlsx"
ruta_2022_2023 = RAW_DATA_DIR / "Totalizadores Planta 2022_2023.xlsx"

# Consultamos los nombres de las hojas de cada archivo con pandas
hojas_2020_2022 = pd.ExcelFile(ruta_2020_2022).sheet_names
hojas_2021_2023 = pd.ExcelFile(ruta_2021_2023).sheet_names
hojas_2022_2023 = pd.ExcelFile(ruta_2022_2023).sheet_names

print("Cantidad de hojas en 2020_2022:", len(hojas_2020_2022))
print("Cantidad de hojas en 2021_2023:", len(hojas_2021_2023))
print("Cantidad de hojas en 2022_2023:", len(hojas_2022_2023))

print("\nLista de hojas en 2020_2022:")
print(hojas_2020_2022)

# Vemos que hoja tiene de mas el archivo 2021_2023 respecto a los otros dos
hoja_diferente = set(hojas_2021_2023) - set(hojas_2020_2022)
print("\nHoja que solo existe en 2021_2023:", hoja_diferente)

Cantidad de hojas en 2020_2022: 15
Cantidad de hojas en 2021_2023: 16
Cantidad de hojas en 2022_2023: 15

Lista de hojas en 2020_2022:
['Consolidado KPI', 'Consolidado Produccion', 'Totalizadores Produccion', 'Consolidado EE', 'Totalizadores Energia', 'Consolidado Agua', 'Totalizadores Agua', 'Consolidado GasVapor', 'Totalizadores Gas y Vapor', 'Consolidado Aire', 'Totalizadores Aire', 'Totalizadores CO2', 'Totalizadores Efluentes', 'Totalizadores Glicol', 'Auxiliar']

Hoja que solo existe en 2021_2023: {'Seguimiento Dia'}


# Fase 1.1

### Vemos el solapamiento de dias

Esta celda carga la hoja principal Consolidado EE (donde está nuestro target Frio (Kw)) de los 3 archivos por separado. Luego hace lo que pide la 1.1.
- Rango de fechas y filas con DIA nulo en cada archivo.
- Qué fechas tienen más de 25 registros horarios dentro de un mismo archivo (jornadas apiladas).
- Cuántas fechas se solapan entre archivos y si el valor de Frio (Kw) al final del día coincide o difiere entre ellos

In [9]:
import pandas as pd


def normalizar_hora(serie):
    s = serie.astype(str).str.strip()
    # Quitar fecha base de Excel si el horario fue leído como datetime
    s = s.str.replace(r"^1899-12-30\s+", "", regex=True)
    # Quitar decimales en horas enteras (ej. '1.0' -> '1')
    s = s.str.replace(r"\.0$", "", regex=True)
    # Asegurar dos dígitos en la hora (ej. '1:00' -> '01:00')
    s = s.str.replace(r"^(\d):", r"0\1:", regex=True)
    # Quitar segundos ':00' para unificar '23:59:00' con '23:59'
    s = s.str.replace(r"^(\d{2}:\d{2}):00$", r"\1", regex=True)
    return s


def cargar_y_limpiar_excel(ruta_archivo, nombre_etiqueta, columnas_clave=["DIA", "HORA"]):
    xls = pd.ExcelFile(ruta_archivo)
    lista_dfs = []
    duplicados_intra_hoja = {}

    for hoja in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=hoja)
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]

        if columnas_clave[0] in df.columns and columnas_clave[1] in df.columns:
            df = df.dropna(subset=columnas_clave).copy()
            df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.normalize()
            df["HORA"] = normalizar_hora(df["HORA"])
            df = df.dropna(subset=["DIA"])

            # Eliminar bloques clonados por meses de 31 días manteniendo el primer bloque real
            n_dup = df.duplicated(subset=columnas_clave, keep="first").sum()
            if n_dup > 0:
                duplicados_intra_hoja[hoja] = int(n_dup)
                df = df.drop_duplicates(subset=columnas_clave, keep="first")

            nuevos_nombres = {
                col: f"{col}_{hoja}"
                for col in df.columns
                if col not in columnas_clave
            }
            df = df.rename(columns=nuevos_nombres)
            lista_dfs.append(df.set_index(columnas_clave))

    print(f"[{nombre_etiqueta}] Hojas procesadas: {len(lista_dfs)} | "
          f"Filas duplicadas intra-hoja eliminadas: {sum(duplicados_intra_hoja.values())}")

    if not lista_dfs:
        return pd.DataFrame()

    df_unificado = pd.concat(lista_dfs, axis=1, join="outer").reset_index()
    df_unificado["_archivo_origen"] = nombre_etiqueta
    return df_unificado


# 1. Cargar y limpiar duplicados intra-Excel de cada archivo
df_2020_2022 = cargar_y_limpiar_excel(ruta_2020_2022, "2020_2022")
df_2021_2023 = cargar_y_limpiar_excel(ruta_2021_2023, "2021_2023")
df_2022_2023 = cargar_y_limpiar_excel(ruta_2022_2023, "2022_2023")

# 2. Auditoría de solapamiento entre archivos (requerido para Fase 1.1)
pares = [
    ("2020_2022", df_2020_2022, "2021_2023", df_2021_2023),
    ("2021_2023", df_2021_2023, "2022_2023", df_2022_2023),
    ("2020_2022", df_2020_2022, "2022_2023", df_2022_2023),
]

print("\n--- AUDITORÍA DE SOLAPAMIENTO ENTRE ARCHIVOS ---")
for nom_a, df_a, nom_b, df_b in pares:
    fechas_comunes = set(df_a["DIA"]).intersection(set(df_b["DIA"]))
    cruce = pd.merge(
        df_a, df_b, on=["DIA", "HORA"], suffixes=(f"_{nom_a}", f"_{nom_b}")
    )
    print(f"{nom_a} vs {nom_b}: {len(fechas_comunes)} fechas en común | "
          f"{len(cruce)} registros (DIA, HORA) solapados")

    # Verificar si el target 'Frio (Kw)_Consolidado EE' coincide en los registros solapados
    col_frio_a = f"Frio (Kw)_Consolidado EE_{nom_a}"
    col_frio_b = f"Frio (Kw)_Consolidado EE_{nom_b}"
    if col_frio_a in cruce.columns and col_frio_b in cruce.columns and len(cruce) > 0:
        s_a = pd.to_numeric(cruce[col_frio_a], errors="coerce")
        s_b = pd.to_numeric(cruce[col_frio_b], errors="coerce")
        iguales = ((s_a - s_b).abs() < 1e-6) | (s_a.isna() & s_b.isna())
        print(f"   -> Coincidencia exacta en 'Frio (Kw)_Consolidado EE': "
              f"{iguales.sum()}/{len(cruce)} registros ({iguales.mean()*100:.2f}%)")

# 3. Unificación final aplicando prioridad: 2022_2023 > 2021_2023 > 2020_2022
df_apilado = pd.concat(
    [df_2022_2023, df_2021_2023, df_2020_2022],
    ignore_index=True
)

df_final = (
    df_apilado.drop_duplicates(subset=["DIA", "HORA"], keep="first")
    .sort_values(["DIA", "HORA"])
    .reset_index(drop=True)
)

# 4. Verificación final de integridad temporal
filas_por_fecha = df_final["DIA"].value_counts()
fechas_mas_25 = filas_por_fecha[filas_por_fecha > 25]

print("\n--- RESUMEN DATASET UNIFICADO (df_final) ---")
print(f"Dimensiones finales: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
print(f"Rango temporal: {df_final['DIA'].min().date()} a {df_final['DIA'].max().date()}")
print(f"Fechas únicas totales: {df_final['DIA'].nunique()}")
print(f"Registros duplicados descartados entre archivos: {len(df_apilado) - len(df_final)}")
print(f"Fechas con más de 25 registros: {len(fechas_mas_25)}")
print("\nRegistros conservados por archivo de origen:")
print(df_final["_archivo_origen"].value_counts())

[2020_2022] Hojas procesadas: 15 | Filas duplicadas intra-hoja eliminadas: 4461
[2021_2023] Hojas procesadas: 16 | Filas duplicadas intra-hoja eliminadas: 4773
[2022_2023] Hojas procesadas: 15 | Filas duplicadas intra-hoja eliminadas: 2945

--- AUDITORÍA DE SOLAPAMIENTO ENTRE ARCHIVOS ---
2020_2022 vs 2021_2023: 254 fechas en común | 6349 registros (DIA, HORA) solapados
   -> Coincidencia exacta en 'Frio (Kw)_Consolidado EE': 6285/6349 registros (98.99%)
2021_2023 vs 2022_2023: 244 fechas en común | 6075 registros (DIA, HORA) solapados
   -> Coincidencia exacta en 'Frio (Kw)_Consolidado EE': 6075/6075 registros (100.00%)
2020_2022 vs 2022_2023: 0 fechas en común | 0 registros (DIA, HORA) solapados

--- RESUMEN DATASET UNIFICADO (df_final) ---
Dimensiones finales: 29756 filas x 422 columnas
Rango temporal: 2020-07-01 a 2023-10-26
Fechas únicas totales: 1190
Registros duplicados descartados entre archivos: 12424
Fechas con más de 25 registros: 30

Registros conservados por archivo de ori

voy a suponer por ahora que esto esta bien

In [12]:
import hashlib
import json
from pathlib import Path

# 1. Quitar la columna auxiliar de auditoría antes de exportar
df_exportar = df_final.drop(columns=["_archivo_origen"], errors="ignore")

# 2. Definir ruta de salida y crear carpetas si no existen
ruta_csv = Path("../data/raw/dataset_todas_tablas.csv")
ruta_csv.parent.mkdir(parents=True, exist_ok=True)

# 3. Exportar a CSV sin índice numérico (evita columnas 'Unnamed: 0' al reabrir)
df_exportar.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Archivo guardado: {ruta_csv} ({df_exportar.shape[0]} filas x {df_exportar.shape[1]} columnas)")


# 4. Calcular SHA-256 por bloques y actualizar data/checksums.json
def calcular_sha256(ruta_archivo, tamano_bloque=65536):
    sha256 = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(tamano_bloque), b""):
            sha256.update(bloque)
    return sha256.hexdigest()


ruta_checksums = Path("../data/checksums.json")
checksums = (
    json.loads(ruta_checksums.read_text(encoding="utf-8"))
    if ruta_checksums.exists()
    else {}
)
checksums[ruta_csv.as_posix()] = calcular_sha256(ruta_csv)

ruta_checksums.write_text(json.dumps(checksums, indent=4), encoding="utf-8")
print(f"SHA-256 registrado en {ruta_checksums}: {checksums[ruta_csv.as_posix()]}")

Archivo guardado: ..\data\raw\dataset_todas_tablas.csv (29756 filas x 421 columnas)
SHA-256 registrado en ..\data\checksums.json: ae0eccf813e4758bc487cb0bd962358e1c1f6eb0d7b0f5a3687e2243f47e3121


In [14]:
df_final.shape

(29756, 422)

Por que hay fechas con 50 o 100 reigstros?:
- 30 de abril, 30 de junio, 30 de septiembre, 30 de noviembre  tienen 50 filas(2×25). Son exactamente los 4 meses del año que tienen 30 días en lugar de 31.
- 28 de febrero: tiene 100 filas (4x25). Es el unico mes que tiene 28 dias, le faltan los dias 29,30,31 y 3 dias x 25hs =75 filas extras.

Entiendo que la explicacion de esto es que quien armo la planilla de excel en la plana, diseño todos los meses como si tuvieran 31 dias de 25hs. En los meses de 30 dias, el dia 31 quedo etiquetado con la fecha del dia 30... Es una suposicion del problema inicial del dataset.

- Cuántos días tienen registro a las 23:00 vs. 23:59 y cuál es su diferencia promedio en kW.

In [15]:
import pandas as pd

# Si estás en una notebook nueva, cargá el CSV generado:
# df_final = pd.read_csv("data/processed/dataset_unificado_inicial.csv")

df_aux = df_final.copy()
df_aux["DIA"] = pd.to_datetime(df_aux["DIA"]).dt.normalize()
df_aux["HORA"] = df_aux["HORA"].astype(str).str.strip()

# Detectar fechas sin registro '23:59'
todas_las_fechas = set(df_aux["DIA"].unique())
fechas_con_2359 = set(df_aux.loc[df_aux["HORA"] == "23:59", "DIA"].unique())
fechas_sin_2359 = sorted(todas_las_fechas - fechas_con_2359)

print(f"Total de fechas únicas: {len(todas_las_fechas)}")
print(f"Fechas CON 23:59: {len(fechas_con_2359)}")
print(f"Fechas SIN 23:59: {len(fechas_sin_2359)} ({len(fechas_sin_2359) / len(todas_las_fechas) * 100:.2f}%)")

# Construir tabla de auditoría para esas fechas
df_filtrado = df_aux[df_aux["DIA"].isin(fechas_sin_2359)].sort_values(["DIA", "HORA"])

col_frio = next(
    (c for c in df_aux.columns if "frio" in c.lower() and "consolidado ee" in c.lower()),
    None
)

agg_dict = {
    "cantidad_registros": ("HORA", "count"),
    "primera_hora": ("HORA", "first"),
    "ultima_hora": ("HORA", "last"),
    "horas_registradas": ("HORA", list),
}
if col_frio:
    agg_dict["ultimo_frio_kw"] = (col_frio, "last")

resumen_sin_2359 = df_filtrado.groupby("DIA", as_index=False).agg(**agg_dict)

print("\n--- DISTRIBUCIÓN POR ÚLTIMA HORA DISPONIBLE ---")
print(resumen_sin_2359["ultima_hora"].value_counts())

print("\n--- DETALLE COMPLETO DE FECHAS SIN 23:59 ---")
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(resumen_sin_2359)

Total de fechas únicas: 1190
Fechas CON 23:59: 1179
Fechas SIN 23:59: 11 (0.92%)

--- DISTRIBUCIÓN POR ÚLTIMA HORA DISPONIBLE ---
ultima_hora
23:59:01    6
23:00       2
19:00       1
16:00       1
07:00       1
Name: count, dtype: int64

--- DETALLE COMPLETO DE FECHAS SIN 23:59 ---


,DIA,cantidad_registros,primera_hora,ultima_hora,horas_registradas,ultimo_frio_kw
0,2021-07-13,23,00:00,23:00,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00:01, 07:00, 08:00, 09:00, 10:00:01, 11:00, 12:00, 13:00, 14:00, 15:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00]",33704.0
1,2021-08-25,25,00:00,23:59:01,"[00:00, 01:00:01, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00:01, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00:01, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00:01, 22:00, 23:00, 23:59:01]",37122.0
2,2022-03-02,24,00:00,23:00,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00]",19294.0
3,2022-04-05,25,00:00,23:59:01,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00:01, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00:01, 19:00, 20:00, 21:00, 22:00, 23:00, 23:59:01]",22573.0
4,2022-04-29,25,00:00,23:59:01,"[00:00, 01:00:01, 02:00:01, 03:00, 04:00, 05:00, 06:00, 07:00:01, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00, 23:59:01]",31316.0
5,2022-12-08,25,00:00,23:59:01,"[00:00, 01:00, 02:00:01, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00:01, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00, 23:59:01]",21663.0
6,2023-02-23,25,00:00,23:59:01,"[00:00, 01:00:01, 02:00, 03:00, 04:00:01, 05:00, 06:00, 07:00, 08:00, 09:00:01, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00:01, 19:00, 20:00, 21:00, 22:00, 23:00, 23:59:01]",20688.0
7,2023-04-13,20,00:00,19:00,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00]",14315.0
8,2023-04-19,17,00:00,16:00,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00]",13439.0
9,2023-07-07,25,00:00,23:59:01,"[00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00:01, 08:00, 09:00, 10:00:01, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00, 23:59:01]",22348.0


In [18]:
import hashlib
import json
from pathlib import Path
import pandas as pd

# 1. Normalizar segundos residuales (ej. '23:59:01' -> '23:59', '01:00:01' -> '01:00')
df_final["HORA"] = (
    df_final["HORA"]
    .astype(str)
    .str.strip()
    .str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)
)

# Evitar colisiones si algún minuto tuviera registro en :00 y :01 simultáneamente
df_final = df_final.drop_duplicates(subset=["DIA", "HORA"], keep="first")

# 2. Identificar fechas con cierre válido a las '23:59'
fechas_validas = df_final.loc[df_final["HORA"] == "23:59", "DIA"].unique()
fechas_eliminadas = sorted(set(df_final["DIA"].unique()) - set(fechas_validas))

print(f"Fechas eliminadas ({len(fechas_eliminadas)}):")
for f in fechas_eliminadas:
    print(f"  - {pd.Timestamp(f).strftime('%Y-%m-%d')}")

# 3. Filtrar df_final dejando únicamente las fechas con cierre '23:59'
df_final = (
    df_final[df_final["DIA"].isin(fechas_validas)]
    .sort_values(["DIA", "HORA"])
    .reset_index(drop=True)
)

print(f"\nFilas finales tras limpieza: {len(df_final)} | Fechas únicas: {df_final['DIA'].nunique()}")

# 4. Sobrescribir el CSV unificado y actualizar su SHA-256 en checksums.json
df_exportar = df_final.drop(columns=["_archivo_origen"], errors="ignore")
ruta_csv = Path("../data/processed/dataset_unificado_inicial.csv")
df_exportar.to_csv(ruta_csv, index=False, encoding="utf-8")

def calcular_sha256(ruta_archivo, tamano_bloque=65536):
    sha256 = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(tamano_bloque), b""):
            sha256.update(bloque)
    return sha256.hexdigest()

ruta_checksums = Path("../data/checksums.json")
checksums = json.loads(ruta_checksums.read_text(encoding="utf-8")) if ruta_checksums.exists() else {}
checksums[ruta_csv.as_posix()] = calcular_sha256(ruta_csv)
ruta_checksums.write_text(json.dumps(checksums, indent=4), encoding="utf-8")

Fechas eliminadas (0):

Filas finales tras limpieza: 29632 | Fechas únicas: 1185


677

In [16]:
import pandas as pd

# Si estás en una notebook nueva, descomentar la siguiente línea:
# df_final = pd.read_csv("data/processed/dataset_unificado_inicial.csv")

fechas_presentes = (
    pd.to_datetime(df_final["DIA"], errors="coerce")
    .dt.normalize()
    .dropna()
    .unique()
)
fechas_presentes = pd.DatetimeIndex(sorted(fechas_presentes))

fecha_min = fechas_presentes.min()
fecha_max = fechas_presentes.max()

# Calendario diario completo entre el mínimo y el máximo
calendario_completo = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Diferencia entre el calendario completo y las fechas existentes
fechas_faltantes = calendario_completo.difference(fechas_presentes)

print(f"Fecha mínima:              {fecha_min.date()}")
print(f"Fecha máxima:              {fecha_max.date()}")
print(f"Días calendario esperados: {len(calendario_completo)}")
print(f"Días únicos presentes:     {len(fechas_presentes)}")
print(f"Días faltantes totales:    {len(fechas_faltantes)} ({len(fechas_faltantes) / len(calendario_completo) * 100:.2f}%)")

if len(fechas_faltantes) > 0:
    # Agrupar días faltantes consecutivos en rangos continuos
    s_faltantes = pd.Series(fechas_faltantes)
    grupos = (s_faltantes.diff() != pd.Timedelta(days=1)).cumsum()
    tramos = (
        s_faltantes.groupby(grupos)
        .agg(
            inicio="min",
            fin="max",
            dias_consecutivos="count",
        )
        .reset_index(drop=True)
    )
    tramos["inicio"] = tramos["inicio"].dt.strftime("%Y-%m-%d")
    tramos["fin"] = tramos["fin"].dt.strftime("%Y-%m-%d")

    print("\n--- TRAMOS DE DÍAS FALTANTES ---")
    with pd.option_context("display.max_rows", None, "display.width", None):
        display(tramos)

    print("\n--- LISTA COMPLETA DE FECHAS FALTANTES ---")
    print([d.strftime("%Y-%m-%d") for d in fechas_faltantes])
else:
    print("\nNo faltan días entre la fecha mínima y máxima.")

Fecha mínima:              2020-07-01
Fecha máxima:              2023-10-26
Días calendario esperados: 1213
Días únicos presentes:     1190
Días faltantes totales:    23 (1.90%)

--- TRAMOS DE DÍAS FALTANTES ---


,inicio,fin,dias_consecutivos
0,2020-08-29,2020-08-31,3
1,2020-10-31,2020-10-31,1
2,2020-12-31,2020-12-31,1
3,2021-03-31,2021-03-31,1
4,2021-05-31,2021-05-31,1
5,2021-08-29,2021-08-31,3
6,2021-10-31,2021-10-31,1
7,2021-12-31,2021-12-31,1
8,2022-03-31,2022-03-31,1
9,2022-05-31,2022-05-31,1



--- LISTA COMPLETA DE FECHAS FALTANTES ---
['2020-08-29', '2020-08-30', '2020-08-31', '2020-10-31', '2020-12-31', '2021-03-31', '2021-05-31', '2021-08-29', '2021-08-30', '2021-08-31', '2021-10-31', '2021-12-31', '2022-03-31', '2022-05-31', '2022-08-29', '2022-08-30', '2022-08-31', '2022-10-31', '2022-12-31', '2023-03-31', '2023-05-31', '2023-08-30', '2023-08-31']


## ANALISIS CONSOLIDADOS KPI

In [19]:
import pandas as pd


def normalizar_hora(serie):
    s = serie.astype(str).str.strip()
    s = s.str.replace(r"^1899-12-30\s+", "", regex=True)
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^(\d):", r"0\1:", regex=True)
    s = s.str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)
    return s


def cargar_hoja_unificada(
    archivos_priorizados,
    nombre_hoja,
    fechas_validas=None,
    columnas_clave=["DIA", "HORA"],
):
    """
    Importa una hoja específica desde múltiples archivos Excel ordenados de mayor
    a menor prioridad, limpiando duplicados y restringiendo a fechas válidas.
    """
    lista_dfs = []
    hoja_objetivo = nombre_hoja.strip()

    for etiqueta, ruta in archivos_priorizados:
        xls = pd.ExcelFile(ruta)
        mapa_hojas = {h.strip(): h for h in xls.sheet_names}

        if hoja_objetivo not in mapa_hojas:
            raise ValueError(
                f"La hoja '{hoja_objetivo}' no existe en '{etiqueta}'. "
                f"Hojas disponibles: {xls.sheet_names}"
            )

        df = pd.read_excel(xls, sheet_name=mapa_hojas[hoja_objetivo])
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]

        faltantes = [c for c in columnas_clave if c not in df.columns]
        if faltantes:
            raise ValueError(
                f"Faltan columnas clave {faltantes} en la hoja '{hoja_objetivo}' ({etiqueta})."
            )

        # Limpieza y estandarización de claves temporales
        df = df.dropna(subset=columnas_clave).copy()
        df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.normalize()
        df["HORA"] = normalizar_hora(df["HORA"])
        df = df.dropna(subset=["DIA"])

        # Eliminar duplicados intra-hoja (meses de 31 días y colisiones :01)
        df = df.drop_duplicates(subset=columnas_clave, keep="first")
        df["_archivo_origen"] = etiqueta
        lista_dfs.append(df)

    # Unificar aplicando prioridad según el orden de la lista
    df_unificado = pd.concat(lista_dfs, ignore_index=True)
    df_unificado = df_unificado.drop_duplicates(subset=columnas_clave, keep="first")

    # Filtrar únicamente los días válidos (con cierre 23:59)
    if fechas_validas is not None:
        fechas_norm = pd.to_datetime(pd.Index(fechas_validas)).normalize()
        df_unificado = df_unificado[df_unificado["DIA"].isin(fechas_norm)]
    else:
        dias_2359 = df_unificado.loc[df_unificado["HORA"] == "23:59", "DIA"].unique()
        df_unificado = df_unificado[df_unificado["DIA"].isin(dias_2359)]

    return df_unificado.sort_values(columnas_clave).reset_index(drop=True)


# Lista ordenada estrictamente del archivo más reciente al más antiguo
archivos_priorizados = [
    ("2022_2023", ruta_2022_2023),
    ("2021_2023", ruta_2021_2023),
    ("2020_2022", ruta_2020_2022),
]

# Reutilizar las fechas limpias de df_final si está en memoria
fechas_filtro = df_final["DIA"].unique() if "df_final" in globals() else None

# Importar únicamente 'Consolidado KPI'
df_kpi = cargar_hoja_unificada(
    archivos_priorizados=archivos_priorizados,
    nombre_hoja="Consolidado KPI",
    fechas_validas=fechas_filtro,
)

print(f"Dimensiones df_kpi: {df_kpi.shape[0]} filas x {df_kpi.shape[1]} columnas")
print(f"Fechas únicas: {df_kpi['DIA'].nunique()} | Rango: {df_kpi['DIA'].min().date()} a {df_kpi['DIA'].max().date()}")
display(df_kpi.head())

Dimensiones df_kpi: 29604 filas x 114 columnas
Fechas únicas: 1185 | Rango: 2020-07-01 a 2023-10-25


,DIA,HORA,EE Planta / Hl,EE Elaboracion / Hl,EE Bodega / Hl,EE Cocina / Hl,EE Envasado / Hl,EE Linea 2 / Hl,EE Linea 3 / Hl,EE Linea 4 / Hl,...,Meta Aire Cocina,Meta Aire Bodega,Meta Aire Envasado,Meta Aire L2,Meta Aire L3,Meta Aire L4,Meta Aire L5,Meta Aire Servicios,_archivo_origen,
0,2020-07-01,00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020_2022,NaN
1,2020-07-01,01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020_2022,NaN
2,2020-07-01,02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020_2022,NaN
3,2020-07-01,03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020_2022,NaN
4,2020-07-01,04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020_2022,NaN


In [20]:
claves_final = set(zip(df_final["DIA"], df_final["HORA"]))
claves_kpi = set(zip(df_kpi["DIA"], df_kpi["HORA"]))
faltantes = sorted(claves_final - claves_kpi)

df_diff = df_final[
    df_final.set_index(["DIA", "HORA"]).index.isin(faltantes)
].copy()

print(f"Filas en df_final pero no en df_kpi: {len(df_diff)}")
display(df_diff[["DIA", "HORA"]])

Filas en df_final pero no en df_kpi: 28


,DIA,HORA
1590,2020-09-05,16:00
8948,2021-07-01,00:00
8949,2021-07-01,01:00
9442,2021-07-21,1900-01-01 00:00:00
9443,2021-07-21,1900-01-01 01:00:00
9444,2021-07-21,1900-01-01 02:00:00
9445,2021-07-21,1900-01-01 03:00:00
9446,2021-07-21,1900-01-01 04:00:00
9447,2021-07-21,1900-01-01 05:00:00
9448,2021-07-21,1900-01-01 06:00:00


Yendo a los datos:
1590	2020-09-05	16:00 -> Esta copiado dos veces las 15:00
8948	2021-07-01	00:00 y 8949	2021-07-01	01:00 -> El contador arranca a las 02:00
21832	2022-12-10	14:00 -> Las 14:00 estan anotadas como 16:00 (hace 16->15->16)


11-12-2022 No tiene datos (justo pase por ahi en el csv)

Recaaaargo con otro codigo

In [22]:
import pandas as pd

archivos = [
    ("2020_2022", ruta_2020_2022),
    ("2021_2023", ruta_2021_2023),
    ("2022_2023", ruta_2022_2023),
]

for etiqueta, ruta in archivos:
    xls = pd.ExcelFile(ruta)
    for hoja in xls.sheet_names:
        df_raw = pd.read_excel(xls, sheet_name=hoja)
        df_raw.columns = df_raw.columns.str.strip()
        if "DIA" not in df_raw.columns or "HORA" not in df_raw.columns:
            continue

        df_raw = df_raw.dropna(subset=["DIA", "HORA"]).copy()
        df_raw["DIA"] = pd.to_datetime(df_raw["DIA"], errors="coerce").dt.normalize()
        s_hora = df_raw["HORA"].astype(str).str.strip()

        # 1) Prefijos de fecha distintos de 1899-12-30 (ej. 1900-01-01)
        prefijos_raros = df_raw[
            s_hora.str.match(r"^\d{4}-\d{2}-\d{2}\s+")
            & ~s_hora.str.startswith("1899-12-30")
        ]
        if not prefijos_raros.empty:
            fechas_afectadas = prefijos_raros["DIA"].dt.strftime("%Y-%m-%d").unique().tolist()
            print(f"[{etiqueta} | {hoja}] Prefijo de fecha anómalo en HORA ({len(prefijos_raros)} filas) -> Fechas: {fechas_afectadas}")

        # 2) Duplicados consecutivos en el mismo día (typos como 15:00 -> 15:00)
        h_norm = (
            s_hora.str.replace(r"^\d{4}-\d{2}-\d{2}\s+", "", regex=True)
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"^(\d):", r"0\1:", regex=True)
            .str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)
        )
        mismo_dia = df_raw["DIA"] == df_raw["DIA"].shift(1)
        dup_seguido = df_raw[mismo_dia & (h_norm == h_norm.shift(1))]
        if not dup_seguido.empty:
            for _, r in dup_seguido.iterrows():
                print(f"[{etiqueta} | {hoja}] Hora repetida consecutiva: {r['DIA'].date()} -> {r['HORA']}")

[2020_2022 | Consolidado KPI] Hora repetida consecutiva: 2020-09-05 -> 15:00:00
[2020_2022 | Consolidado KPI] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Consolidado Produccion] Hora repetida consecutiva: 2020-09-05 -> 15:00:00
[2020_2022 | Consolidado Produccion] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Totalizadores Produccion] Hora repetida consecutiva: 2020-09-05 -> 15:00:00
[2020_2022 | Totalizadores Produccion] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Consolidado EE] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Totalizadores Energia] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Consolidado Agua] Prefijo de fecha anómalo en HORA (24 filas) -> Fechas: ['2021-07-21']
[2020_2022 | Consolidado Agua] Hora repetida consecutiva: 2021-06-30 -> 23:59:59
[2020_2022 | Totalizadores Agua] Prefijo de fecha anómalo en HORA (24 filas) -> Fechas: ['2021-07-21']
[2020_2022 | Totalizadores Agua] Hora 

In [23]:
def limpiar_horas_y_cierre(df):
    """
    1. Elimina prefijos de fecha (1899-12-30, 1900-01-01, etc.) y estandariza HH:MM(:SS).
    2. Si un DIA tiene tanto '23:59' (o '23:59:00'/'23:59:01') como '23:59:59', elimina '23:59:59'.
    3. Unifica el resto de segundos residuales al formato final 'HH:MM'.
    """
    df = df.copy()
    s = df["HORA"].astype(str).str.strip()

    # Quitar cualquier prefijo de fecha (ej. '1899-12-30 ', '1900-01-01 ')
    s = s.str.replace(r"^\d{4}-\d{2}-\d{2}\s+", "", regex=True)
    # Quitar decimales en horas enteras (ej. '1.0' -> '1')
    s = s.str.replace(r"\.0$", "", regex=True)
    # Asegurar dos dígitos en la hora (ej. '1:00' -> '01:00')
    s = s.str.replace(r"^(\d):", r"0\1:", regex=True)
    # Normalizar ':00' y ':01' a 'HH:MM', pero conservar ':59' para poder filtrarlo
    s = s.str.replace(r"^(\d{2}:\d{2}):(00|01)$", r"\1", regex=True)
    df["HORA"] = s

    # Días que ya cuentan con registro a las '23:59'
    dias_con_2359 = set(df.loc[df["HORA"] == "23:59", "DIA"])

    # Borrar '23:59:59' únicamente en esos días
    mask_eliminar = (df["HORA"] == "23:59:59") & (df["DIA"].isin(dias_con_2359))
    df = df[~mask_eliminar].copy()

    # Si algún día tenía solo '23:59:59' (sin '23:59'), conservarlo como '23:59'
    df["HORA"] = df["HORA"].str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)
    return df

In [25]:
import pandas as pd
import numpy as np

HORAS_CANONICAS = [f"{h:02d}:00" for h in range(24)] + ["23:59"]

def limpiar_horas_y_cierre(df):
    """Limpia fechas incrustadas, recorta 23:59:59 y normaliza segundos."""
    df = df.copy()
    s = df["HORA"].astype(str).str.strip()
    s = s.str.replace(r"^\d{4}-\d{2}-\d{2}\s+", "", regex=True)
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^(\d):", r"0\1:", regex=True)
    
    mask_235959 = s.str.startswith("23:59:59")
    df = df[~mask_235959].copy()
    s = s[~mask_235959]
    
    df["HORA"] = s.str.replace(r"^(\d{2}:\d{2}):\d{2}$", r"\1", regex=True)
    return df

def reparar_typos_horarios(df):
    """Corrige errores de tipeo humano manteniendo la estructura secuencial."""
    df = df.copy()
    horas_corregidas = []
    
    for dia, grupo in df.groupby("DIA", sort=False):
        h_list = grupo["HORA"].tolist()
        if len(h_list) == 25 and h_list[0] == "00:00" and h_list[-1] == "23:59":
            if h_list != HORAS_CANONICAS:
                h_list = HORAS_CANONICAS.copy()
        else:
            for i in range(1, len(h_list) - 1):
                prev_h, curr_h, next_h = h_list[i - 1], h_list[i], h_list[i + 1]
                if prev_h in HORAS_CANONICAS[:-1] and next_h in HORAS_CANONICAS[:-1]:
                    idx_prev = HORAS_CANONICAS.index(prev_h)
                    idx_next = HORAS_CANONICAS.index(next_h)
                    if idx_next - idx_prev == 2:
                        esperada = HORAS_CANONICAS[idx_prev + 1]
                        if curr_h != esperada:
                            h_list[i] = esperada
        horas_corregidas.extend(h_list)
        
    df["HORA"] = horas_corregidas
    return df

# 1. Cargar y limpiar Consolidado KPI archivo por archivo
archivos_priorizados = [
    ("2022_2023", ruta_2022_2023),
    ("2021_2023", ruta_2021_2023),
    ("2020_2022", ruta_2020_2022),
]

lista_dfs_kpi = []
for etiqueta, ruta in archivos_priorizados:
    xls = pd.ExcelFile(ruta)
    # Manejar posibles espacios en los nombres de las hojas
    mapa_hojas = {h.strip(): h for h in xls.sheet_names}
    
    if "Consolidado KPI" in mapa_hojas:
        df = pd.read_excel(xls, sheet_name=mapa_hojas["Consolidado KPI"])
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]
        
        df = df.dropna(subset=["DIA", "HORA"]).copy()
        df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.normalize()
        df = df.dropna(subset=["DIA"])
        
        df = limpiar_horas_y_cierre(df)
        
        es_nuevo_bloque = (df["HORA"] == "00:00") & (df["DIA"] == df["DIA"].shift(1))
        id_bloque = es_nuevo_bloque.groupby(df["DIA"]).cumsum()
        df = df[id_bloque == 0].copy()
        
        df = reparar_typos_horarios(df)
        df = df.drop_duplicates(subset=["DIA", "HORA"], keep="first")
        df["_archivo_origen"] = etiqueta
        lista_dfs_kpi.append(df)

# 2. Unificar respetando la prioridad
df_kpi_unificado = pd.concat(lista_dfs_kpi, ignore_index=True)
df_kpi_unificado = (
    df_kpi_unificado.drop_duplicates(subset=["DIA", "HORA"], keep="first")
    .sort_values(["DIA", "HORA"])
    .reset_index(drop=True)
)

# 3. Auditoría Estructural (00:00, 23:59 y cantidad de horas)
resumen_kpi = df_kpi_unificado.groupby("DIA").agg(
    n_registros=("HORA", "count"),
    tiene_0000=("HORA", lambda s: "00:00" in s.values),
    tiene_2359=("HORA", lambda s: "23:59" in s.values),
)

mascara_valida = (
    (resumen_kpi["tiene_0000"]) & 
    (resumen_kpi["tiene_2359"]) & 
    (resumen_kpi["n_registros"] >= 24)
)

dias_validos_kpi = resumen_kpi[mascara_valida].index
dias_eliminados_kpi = resumen_kpi[~mascara_valida]

# 4. Reporte
print("=== REPORTE DE AUDITORÍA: Consolidado KPI ===")
print(f"Total de días únicos evaluados: {len(resumen_kpi)}")
print(f"Días que cumplen la estructura: {len(dias_validos_kpi)}")
print(f"Días eliminados:                {len(dias_eliminados_kpi)}")

if len(dias_eliminados_kpi) > 0:
    print("\n--- Detalle de días eliminados ---")
    for dia, row in dias_eliminados_kpi.iterrows():
        fallos = []
        if not row['tiene_0000']: fallos.append("Falta 00:00")
        if not row['tiene_2359']: fallos.append("Falta 23:59")
        if row['n_registros'] < 24: fallos.append(f"Registros insuficientes ({row['n_registros']})")
        
        print(f"  - {dia.strftime('%Y-%m-%d')}: {', '.join(fallos)}")

# 5. Filtrar el dataframe definitivo
df_kpi_final = df_kpi_unificado[df_kpi_unificado["DIA"].isin(dias_validos_kpi)].reset_index(drop=True)

=== REPORTE DE AUDITORÍA: Consolidado KPI ===
Total de días únicos evaluados: 1190
Días que cumplen la estructura: 1180
Días eliminados:                10

--- Detalle de días eliminados ---
  - 2021-07-01: Falta 00:00, Registros insuficientes (23)
  - 2021-07-13: Falta 23:59, Registros insuficientes (23)
  - 2021-11-01: Registros insuficientes (23)
  - 2022-03-02: Falta 23:59
  - 2023-01-17: Registros insuficientes (16)
  - 2023-02-28: Falta 23:59
  - 2023-04-13: Falta 23:59, Registros insuficientes (20)
  - 2023-04-19: Falta 23:59, Registros insuficientes (17)
  - 2023-07-01: Falta 00:00, Registros insuficientes (23)
  - 2023-10-26: Falta 23:59, Registros insuficientes (8)


In [26]:
import pandas as pd
import numpy as np

# Reutilizamos las funciones de limpieza estructural definidas anteriormente
# Asegúrate de tener en memoria: limpiar_horas_y_cierre y reparar_typos_horarios

archivos_priorizados = [
    ("2022_2023", ruta_2022_2023),
    ("2021_2023", ruta_2021_2023),
    ("2020_2022", ruta_2020_2022),
]

# 1. Cargar y limpiar estructuralmente Consolidado EE
lista_dfs_ee = []
for etiqueta, ruta in archivos_priorizados:
    xls = pd.ExcelFile(ruta)
    mapa_hojas = {h.strip(): h for h in xls.sheet_names}
    
    if "Consolidado EE" in mapa_hojas:
        df = pd.read_excel(xls, sheet_name=mapa_hojas["Consolidado EE"])
        df.columns = df.columns.str.strip()
        df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False)]
        
        df = df.dropna(subset=["DIA", "HORA"]).copy()
        df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.normalize()
        df = df.dropna(subset=["DIA"])
        
        df = limpiar_horas_y_cierre(df)
        
        es_nuevo_bloque = (df["HORA"] == "00:00") & (df["DIA"] == df["DIA"].shift(1))
        id_bloque = es_nuevo_bloque.groupby(df["DIA"]).cumsum()
        df = df[id_bloque == 0].copy()
        
        df = reparar_typos_horarios(df)
        df = df.drop_duplicates(subset=["DIA", "HORA"], keep="first")
        df["_archivo_origen"] = etiqueta
        lista_dfs_ee.append(df)

df_ee_unificado = pd.concat(lista_dfs_ee, ignore_index=True)
df_ee_unificado = (
    df_ee_unificado.drop_duplicates(subset=["DIA", "HORA"], keep="first")
    .sort_values(["DIA", "HORA"])
    .reset_index(drop=True)
)

# 2. Manejo de "Datos Feos" (textos de error como #VALUE!)
# Forzamos la conversión a numérico; los errores de Excel se convertirán en NaN
columnas_variables = [c for c in df_ee_unificado.columns if c not in ["DIA", "HORA", "_archivo_origen"]]
for col in columnas_variables:
    df_ee_unificado[col] = pd.to_numeric(df_ee_unificado[col], errors="coerce")

# 3. Auditoría Estructural y Física
reporte_ee = []
TOL = 1e-3  # Tolerancia para redondeo de coma flotante

for dia, grupo in df_ee_unificado.groupby("DIA", sort=True):
    grupo = grupo.sort_values("HORA")
    horas_presentes = grupo["HORA"].tolist()
    n_registros = len(horas_presentes)
    tiene_0000 = "00:00" in horas_presentes
    tiene_2359 = "23:59" in horas_presentes
    
    # Evaluar estructura temporal básica (días incompletos)
    estructura_valida = (n_registros >= 24) and tiene_0000 and tiene_2359
    
    # Extraer serie del Target
    s_frio = grupo.set_index("HORA")["Frio (Kw)"].dropna() if "Frio (Kw)" in grupo.columns else pd.Series(dtype=float)
    
    if len(s_frio) == 0:
        frio_vacio = True
        frio_monotono = frio_pico_01 = frio_sin_reinicio = frio_imposible = negativo_frio = False
        val_2359 = np.nan
    else:
        frio_vacio = False
        diffs_frio = s_frio.diff().dropna()
        
        # ¿Tiene valores negativos? (Imposibilidad física)
        negativo_frio = bool((s_frio < 0).any())
        
        # ¿Es monótona creciente? (No puede disminuir respecto a la hora anterior)
        frio_monotono = bool((diffs_frio >= -TOL).all())
        
        # Pico gigante a las 01:00 que cae en 02:00
        val_00 = s_frio.get("00:00", np.nan)
        val_01 = s_frio.get("01:00", np.nan)
        val_02 = s_frio.get("02:00", np.nan)
        frio_pico_01 = bool(pd.notna(val_01) and pd.notna(val_02) and (val_01 > val_02 + TOL))
        
        # ¿No se reinició a las 00:00?
        frio_sin_reinicio = bool(pd.notna(val_00) and ((val_00 > 5000) or (pd.notna(val_01) and val_00 > val_01 + TOL)))
        
        # ¿Valor final irreal? Rango típico: 0 - 35,000 kW (tolerancia hasta 60k por si hubo picos de producción)
        val_2359 = s_frio.get("23:59", np.nan)
        frio_imposible = bool(pd.isna(val_2359) or (val_2359 <= 0) or (val_2359 > 100_000))
        
    reporte_ee.append({
        "DIA": dia,
        "estructura_valida": estructura_valida,
        "frio_vacio": frio_vacio,
        "frio_negativo": negativo_frio,
        "frio_monotono": frio_monotono,
        "frio_pico_01": frio_pico_01,
        "frio_sin_reinicio": frio_sin_reinicio,
        "frio_imposible": frio_imposible,
        "frio_2359_kw": val_2359,
        "n_registros": n_registros
    })

df_auditoria_ee = pd.DataFrame(reporte_ee)

# 4. Clasificación de Días
dias_estructuralmente_invalidos = df_auditoria_ee[~df_auditoria_ee["estructura_valida"]]
dias_fisicamente_invalidos_frio = df_auditoria_ee[
    df_auditoria_ee["estructura_valida"] & 
    (df_auditoria_ee["frio_vacio"] | 
     df_auditoria_ee["frio_negativo"] | 
     df_auditoria_ee["frio_sin_reinicio"] | 
     df_auditoria_ee["frio_imposible"] | 
     (~df_auditoria_ee["frio_monotono"] & ~df_auditoria_ee["frio_pico_01"])) # Permitimos temporalmente el pico 01 si el resto es monótono
]

dias_a_conservar = df_auditoria_ee[
    df_auditoria_ee["estructura_valida"] & 
    ~df_auditoria_ee["DIA"].isin(dias_fisicamente_invalidos_frio["DIA"])
]

df_ee_final = df_ee_unificado[df_ee_unificado["DIA"].isin(dias_a_conservar["DIA"])].reset_index(drop=True)

# 5. Reporte de Consolidado EE
print("=== REPORTE DE AUDITORÍA: Consolidado EE (Enfoque en Frio kW) ===")
print(f"Total de días únicos en los crudos:      {len(df_auditoria_ee)}")
print(f"Días eliminados por estructura rota:     {len(dias_estructuralmente_invalidos)} (Falta 00:00, 23:59 o < 24 registros)")
print(f"Días eliminados por física imposible:    {len(dias_fisicamente_invalidos_frio)} (En variable objetivo)")
print(f"Días válidos retenidos para modelado:    {len(dias_a_conservar)}")

print("\n--- Desglose de anomalías físicas en Frio (Kw) (sobre días con estructura válida) ---")
print(f"  - Todo nulo / '#VALUE!':               {dias_fisicamente_invalidos_frio['frio_vacio'].sum()}")
print(f"  - Valores negativos:                   {dias_fisicamente_invalidos_frio['frio_negativo'].sum()}")
print(f"  - Falla en reinicio diario:            {dias_fisicamente_invalidos_frio['frio_sin_reinicio'].sum()}")
print(f"  - Caídas (no monótonos):               {(~dias_fisicamente_invalidos_frio['frio_monotono'] & ~dias_fisicamente_invalidos_frio['frio_pico_01']).sum()}")
print(f"  - Acumulado total absurdo (>100k):     {dias_fisicamente_invalidos_frio['frio_imposible'].sum()}")

print("\nNota: Los días que presentan el 'pico de las 01:00' pero regresan a la normalidad en 02:00 fueron retenidos si cumplen la monotonía el resto del día, ya que el valor de las 23:59 permanece inalterado para el shift.")

=== REPORTE DE AUDITORÍA: Consolidado EE (Enfoque en Frio kW) ===
Total de días únicos en los crudos:      1190
Días eliminados por estructura rota:     9 (Falta 00:00, 23:59 o < 24 registros)
Días eliminados por física imposible:    22 (En variable objetivo)
Días válidos retenidos para modelado:    1159

--- Desglose de anomalías físicas en Frio (Kw) (sobre días con estructura válida) ---
  - Todo nulo / '#VALUE!':               0
  - Valores negativos:                   0
  - Falla en reinicio diario:            1
  - Caídas (no monótonos):               22
  - Acumulado total absurdo (>100k):     0

Nota: Los días que presentan el 'pico de las 01:00' pero regresan a la normalidad en 02:00 fueron retenidos si cumplen la monotonía el resto del día, ya que el valor de las 23:59 permanece inalterado para el shift.


In [27]:
import pandas as pd
import numpy as np

# Se asume que df_ee_unificado ya está en memoria tras ejecutar el script anterior
reporte_detallado = []
TOL = 1e-3

for dia, grupo in df_ee_unificado.groupby("DIA", sort=True):
    grupo = grupo.sort_values("HORA")
    horas = grupo["HORA"].tolist()
    
    # Extraer el archivo de origen dominante para ese día (tomando el último registro válido)
    archivo_origen = grupo["_archivo_origen"].iloc[-1]
    
    # --- 1. INSTANCIA: ESTRUCTURAL ---
    n_reg = len(horas)
    falta_00 = "00:00" not in horas
    falta_2359 = "23:59" not in horas
    
    motivos_estructurales = []
    if falta_00: motivos_estructurales.append("Falta lectura 00:00")
    if falta_2359: motivos_estructurales.append("Falta lectura de cierre 23:59")
    if n_reg < 24: motivos_estructurales.append(f"Registros insuficientes ({n_reg}/24)")
    
    if motivos_estructurales:
        reporte_detallado.append({
            "DIA": dia,
            "Archivo": archivo_origen,
            "Instancia": "Estructural",
            "Motivo": " | ".join(motivos_estructurales),
            "Valor_2359_kW": np.nan
        })
        continue  # Si la estructura temporal está rota, se aborta la auditoría física
        
    # --- 2. INSTANCIA: FÍSICA (Comportamiento Acumulativo) ---
    if "Frio (Kw)" not in grupo.columns:
        s_frio = pd.Series(dtype=float)
    else:
        s_frio = grupo.set_index("HORA")["Frio (Kw)"].dropna()
    
    motivos_fisicos = []
    val_2359 = s_frio.get("23:59", np.nan)
    
    if len(s_frio) == 0:
        motivos_fisicos.append("Columna sin datos numéricos (puro NaN o '#VALUE!')")
    else:
        val_00 = s_frio.get("00:00", np.nan)
        val_01 = s_frio.get("01:00", np.nan)
        val_02 = s_frio.get("02:00", np.nan)
        
        diffs_frio = s_frio.diff().dropna()
        
        # A. Valores Negativos (Imposibilidad termodinámica y eléctrica)
        if (s_frio < 0).any():
            motivos_fisicos.append("Contiene valores negativos")
            
        # B. Fallo de reinicio diario a las 00:00
        if pd.notna(val_00) and ((val_00 > 5000) or (pd.notna(val_01) and val_00 > val_01 + TOL)):
            motivos_fisicos.append(f"Falla de reinicio: Arranca en {val_00} kW a las 00:00")
            
        # C. Acumulado final absurdo
        if pd.isna(val_2359) or (val_2359 <= 0) or (val_2359 > 100_000):
            motivos_fisicos.append(f"Acumulado 23:59 irreal o nulo ({val_2359} kW)")
            
        # D. Monotonía estricta (excluyendo el falso positivo del pico a las 01:00)
        es_monotono = (diffs_frio >= -TOL).all()
        pico_01 = pd.notna(val_01) and pd.notna(val_02) and (val_01 > val_02 + TOL)
        
        if not es_monotono:
            if pico_01:
                # Verificar si hay más caídas aparte de la caída de 01:00 a 02:00
                diffs_sin_pico = diffs_frio.drop("02:00", errors="ignore")
                if not (diffs_sin_pico >= -TOL).all():
                    motivos_fisicos.append("Múltiples violaciones de monotonía (caídas) además del pico 01:00")
            else:
                motivos_fisicos.append("Violación de monotonía (caídas a mitad del día)")

    if motivos_fisicos:
        reporte_detallado.append({
            "DIA": dia,
            "Archivo": archivo_origen,
            "Instancia": "Física",
            "Motivo": " | ".join(motivos_fisicos),
            "Valor_2359_kW": val_2359
        })

df_dias_borrados = pd.DataFrame(reporte_detallado)

print(f"Total de días bloqueados y eliminados: {len(df_dias_borrados)}")
print("Vista previa del registro:")
display(df_dias_borrados.head(20))

# Exportar a CSV para verificación manual cruzada con los Excel crudos
df_dias_borrados.to_csv("../data/processed/auditoria_dias_borrados_EE.csv", index=False)
print("Archivo exportado: data/processed/auditoria_dias_borrados_EE.csv")

Total de días bloqueados y eliminados: 31
Vista previa del registro:


,DIA,Archivo,Instancia,Motivo,Valor_2359_kW
0,2020-07-11,2020_2022,Física,Falla de reinicio: Arranca en 29113.0 kW a las...,29046.0
1,2020-09-11,2020_2022,Física,Violación de monotonía (caídas a mitad del día),29293.0
2,2021-01-03,2020_2022,Física,Violación de monotonía (caídas a mitad del día),4925.0
3,2021-01-21,2020_2022,Física,Violación de monotonía (caídas a mitad del día),16699.0
4,2021-01-31,2020_2022,Física,Violación de monotonía (caídas a mitad del día),16218.0
5,2021-03-18,2020_2022,Física,Violación de monotonía (caídas a mitad del día),24261.0
6,2021-03-19,2020_2022,Física,Violación de monotonía (caídas a mitad del día),23964.0
7,2021-04-04,2020_2022,Física,Violación de monotonía (caídas a mitad del día),24715.0
8,2021-04-21,2020_2022,Física,Violación de monotonía (caídas a mitad del día),31212.0
9,2021-04-22,2020_2022,Física,Violación de monotonía (caídas a mitad del día),27116.0


Archivo exportado: data/processed/auditoria_dias_borrados_EE.csv


Con esto creo ya estarian las fechas limpitas